<!-- paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 참고 구현입니다. 필요한 한 줄만 확인하고 다시 실습 창으로 돌아가세요.

# 22. Hybrid RAG 평가와 회귀 테스트

`data/practice/rag_queries.jsonl`을 고정 평가셋으로 사용해 sparse+dense
검색, reciprocal rank fusion(RRF),
Recall@k/MRR, no-answer threshold, 증분 index/version metadata, 회귀 리포트를 외부 API 없이 구현합니다.
17·18번을 끝낸 뒤 “좋아 보이는 검색”을 **재현 가능한 수치와 배포 gate**로 바꾸는 실습입니다.


In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 두 창 학습법과 목표

왼쪽 exercise의 TODO를 먼저 구현하고 오른쪽 solution으로 확인하세요.
같은 query라도 sparse/dense/hybrid의 순위가 왜 다른지, answerable과
unanswerable에서 threshold가 어떤 trade-off를 만드는지 기록합니다.
마지막에는 baseline 대비 regression을 자동 판정합니다.


In [ ]:
import hashlib, json, math, re, tempfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer


def find_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data" / "practice" / "rag_queries.jsonl").exists():
            return candidate
    raise RuntimeError("코딩연습 폴더 또는 하위에서 실행하세요.")


ROOT = find_root()
KB_PATH = ROOT / "data" / "raw" / "knowledge_base.jsonl"
EVAL_PATH = ROOT / "data" / "practice" / "rag_queries.jsonl"
print(KB_PATH, EVAL_PATH, sep="\n")

## 1) corpus와 평가셋 계약

평가 query의 `relevant_ids=[]`는 no-answer 사례입니다. 정답 id가 corpus에 없거나 query id가 중복되면 지표가 거짓말하므로
평가 전에 막습니다. 평가 파일은 모델 튜닝용 train set이 아니라 고정 regression set처럼 다룹니다.


In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8-sig") as handle:
        return [json.loads(line) for line in handle if line.strip()]


documents = read_jsonl(KB_PATH)
eval_cases = read_jsonl(EVAL_PATH)
document_ids = [row["id"] for row in documents]
query_ids = [row["query_id"] for row in eval_cases]
assert len(document_ids) == len(set(document_ids))
assert len(query_ids) == len(set(query_ids))
assert all(set(case["relevant_ids"]) <= set(document_ids) for case in eval_cases)
print(
    "documents=",
    len(documents),
    "eval cases=",
    len(eval_cases),
    "no-answer=",
    sum(not c["relevant_ids"] for c in eval_cases),
)

In [ ]:
eval_frame = pd.DataFrame(
    {
        "query_id": c["query_id"],
        "query": c["query"],
        "query_chars": len(c["query"]),
        "relevant_count": len(c["relevant_ids"]),
        "answerable": bool(c["relevant_ids"]),
    }
    for c in eval_cases
)
display(eval_frame)

## 2) sparse index와 version metadata

문자 n-gram TF-IDF를 sparse 기준선으로 만듭니다. index version에는
corpus hash, document 수, embedding 설정을 기록합니다.
단순 날짜보다 content hash가 같은 데이터/설정에서 같은 artifact임을 확인하기 쉽습니다.


In [ ]:
def document_text(row: dict) -> str:
    return " ".join((row["title"], row["category"], *row["keywords"], row["content"]))


def corpus_hash(records: list[dict]) -> str:
    canonical = json.dumps(
        sorted(records, key=lambda r: r["id"]), ensure_ascii=False, sort_keys=True
    )
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]


@dataclass
class RankedHit:
    document_id: str
    rank: int
    score: float
    retriever: str

In [ ]:
class SparseIndex:
    def fit(self, records: list[dict]):
        self.documents = list(records)
        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True
        )
        self.matrix = self.vectorizer.fit_transform(
            [document_text(r) for r in records]
        ).tocsr()
        self.version = {
            "kind": "char_tfidf",
            "schema_version": 1,
            "corpus_hash": corpus_hash(records),
            "document_count": len(records),
            "features": self.matrix.shape[1],
        }
        return self

    def search(self, query: str, top_k: int = 5) -> list[RankedHit]:
        if not query.strip():
            return []
        scores = (self.matrix @ self.vectorizer.transform([query]).T).toarray().ravel()
        order = np.argsort(scores)[::-1][:top_k]
        return [
            RankedHit(self.documents[i]["id"], rank, float(scores[i]), "sparse")
            for rank, i in enumerate(order, 1)
        ]

In [ ]:
sparse_index = SparseIndex().fit(documents)
assert sparse_index.version["document_count"] == len(documents)
print(sparse_index.version)
display(pd.DataFrame(vars(h) for h in sparse_index.search("API 요청에서 401", 3)))

## 3) deterministic dense embedding: TF-IDF + SVD

외부 모델 다운로드 없이 dense vector를 만들기 위해 TF-IDF를 TruncatedSVD로 투영합니다. 이는 의미 학습 모델이 아니라
latent lexical baseline이지만 sparse/dense 결합, 정규화, dimension, index 계약을 재현 가능하게 연습할 수 있습니다.


In [ ]:
class DenseSVDIndex:
    def fit(self, records: list[dict], n_components: int = 10):
        self.documents = list(records)
        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True
        )
        sparse = self.vectorizer.fit_transform([document_text(r) for r in records])
        components = min(n_components, sparse.shape[0] - 1, sparse.shape[1] - 1)
        self.svd = TruncatedSVD(n_components=components, random_state=22)
        vectors = self.svd.fit_transform(sparse).astype(np.float32)
        self.vectors = vectors / np.maximum(
            np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12
        )
        self.version = {
            "kind": "tfidf_svd",
            "schema_version": 1,
            "corpus_hash": corpus_hash(records),
            "document_count": len(records),
            "dimensions": components,
        }
        return self

    def search(self, query: str, top_k: int = 5) -> list[RankedHit]:
        if not query.strip():
            return []
        vector = self.svd.transform(self.vectorizer.transform([query])).astype(
            np.float32
        )[0]
        vector /= max(float(np.linalg.norm(vector)), 1e-12)
        scores = self.vectors @ vector
        order = np.argsort(scores)[::-1][:top_k]
        return [
            RankedHit(self.documents[i]["id"], rank, float(scores[i]), "dense_svd")
            for rank, i in enumerate(order, 1)
        ]

In [ ]:
dense_index = DenseSVDIndex().fit(documents, n_components=10)
assert dense_index.vectors.shape == (len(documents), 10)
assert np.allclose(np.linalg.norm(dense_index.vectors, axis=1), 1.0)
assert dense_index.version["corpus_hash"] == sparse_index.version["corpus_hash"]
display(pd.DataFrame(vars(h) for h in dense_index.search("API 요청에서 401", 3)))

## 4) Reciprocal Rank Fusion

sparse score와 dense score는 척도가 다르므로 직접 더하지 않습니다. RRF는 각 ranking의 `1/(rrf_k + rank)`를 더해
score calibration 없이 순위를 결합합니다. 한 retriever에만 등장한 문서도 포함됩니다.


In [ ]:
def reciprocal_rank_fusion(
    rankings: list[list[RankedHit]], top_k: int = 5, rrf_k: int = 60
) -> list[RankedHit]:
    fused = {}
    for ranking in rankings:
        for hit in ranking:
            fused[hit.document_id] = fused.get(hit.document_id, 0.0) + 1.0 / (
                rrf_k + hit.rank
            )
    ordered = sorted(fused.items(), key=lambda item: (-item[1], item[0]))[:top_k]
    return [
        RankedHit(document_id, rank, score, "hybrid_rrf")
        for rank, (document_id, score) in enumerate(ordered, 1)
    ]


def hybrid_search(query: str, top_k: int = 5) -> list[RankedHit]:
    candidate_k = min(len(documents), max(top_k * 2, 8))
    return reciprocal_rank_fusion(
        [
            sparse_index.search(query, candidate_k),
            dense_index.search(query, candidate_k),
        ],
        top_k=top_k,
    )

In [ ]:
fusion_query = "운송장 조회가 이틀째 그대로예요"
fusion_frame = pd.DataFrame(
    vars(hit)
    for search in (sparse_index.search, dense_index.search, hybrid_search)
    for hit in search(fusion_query, 3)
)
display(fusion_frame)

## 5) Recall@k와 MRR

여러 relevant id가 있으면 하나라도 top-k에 들어오면 Recall hit로 보고,
첫 relevant의 reciprocal rank를 MRR에 사용합니다.
no-answer case는 retrieval ranking metric에서 제외하고 threshold 평가에서 별도로 다룹니다.


In [ ]:
def ranking_metrics(search_fn, cases: list[dict], k: int) -> dict:
    answerable = [case for case in cases if case["relevant_ids"]]
    hits, reciprocal_ranks = 0, []
    for case in answerable:
        ids = [hit.document_id for hit in search_fn(case["query"], k)]
        relevant = set(case["relevant_ids"])
        hits += bool(relevant & set(ids))
        ranks = [
            rank for rank, document_id in enumerate(ids, 1) if document_id in relevant
        ]
        reciprocal_ranks.append(1 / min(ranks) if ranks else 0.0)
    return {
        "recall_at_k": hits / len(answerable),
        "mrr": float(np.mean(reciprocal_ranks)),
    }

In [ ]:
searchers = {
    "sparse": sparse_index.search,
    "dense_svd": dense_index.search,
    "hybrid_rrf": hybrid_search,
}
ranking_report = pd.DataFrame(
    {"retriever": name, "k": k, **ranking_metrics(search_fn, eval_cases, k)}
    for name, search_fn in searchers.items()
    for k in (1, 3, 5)
)
assert ranking_report["recall_at_k"].between(0, 1).all()
display(ranking_report.round(3))

## 6) no-answer threshold

RRF score는 작고 좁은 범위라 no-answer threshold 설명에 불편합니다.
여기서는 hybrid top-1 문서에 대해 sparse/dense cosine의
최댓값을 confidence proxy로 사용합니다. 이 값도 확률은 아니며 validation distribution에서 threshold를 정해야 합니다.


In [ ]:
def hybrid_confidence(query: str, result: list[RankedHit]) -> float:
    if not result:
        return 0.0
    top_id = result[0].document_id
    sparse_scores = {
        h.document_id: h.score for h in sparse_index.search(query, len(documents))
    }
    dense_scores = {
        h.document_id: h.score for h in dense_index.search(query, len(documents))
    }
    return max(sparse_scores.get(top_id, 0.0), dense_scores.get(top_id, 0.0))


def threshold_metrics(threshold: float) -> dict:
    (
        answered,
        correct_answerable,
        answerable_count,
        correct_abstain,
        unanswerable_count,
    ) = 0, 0, 0, 0, 0
    for case in eval_cases:
        result = hybrid_search(case["query"], 5)
        confidence = hybrid_confidence(case["query"], result)
        should_answer = confidence >= threshold
        answered += should_answer
        if case["relevant_ids"]:
            answerable_count += 1
            correct_answerable += (
                should_answer and result[0].document_id in case["relevant_ids"]
            )
        else:
            unanswerable_count += 1
            correct_abstain += not should_answer
    return {
        "threshold": threshold,
        "coverage": answered / len(eval_cases),
        "answerable_accuracy": correct_answerable / answerable_count,
        "no_answer_accuracy": correct_abstain / unanswerable_count,
    }

In [ ]:
threshold_report = pd.DataFrame(
    threshold_metrics(float(t)) for t in np.linspace(0.0, 0.30, 7)
)
POLICY_THRESHOLD = 0.15
display(threshold_report.round(3))

## 7) 증분 update와 version metadata

TF-IDF/SVD는 vocabulary와 global statistics를 학습하므로 진짜
append-only update가 아닙니다. 작은 corpus에서는 새 문서를 합친 뒤
재학습하는 것이 안전합니다. 이를 `update()`라는 API로 감싸되
parent hash, 새 hash, generation을 기록해 lineage를 보존합니다.


In [ ]:
class VersionedHybridIndex:
    def __init__(
        self, records: list[dict], generation: int = 1, parent_hash: str | None = None
    ):
        ids = [row["id"] for row in records]
        if len(ids) != len(set(ids)):
            raise ValueError("document ids must be unique")
        self.documents = list(records)
        self.sparse = SparseIndex().fit(records)
        self.dense = DenseSVDIndex().fit(
            records, n_components=min(10, len(records) - 1)
        )
        self.version = {
            "schema_version": 1,
            "generation": generation,
            "corpus_hash": corpus_hash(records),
            "parent_hash": parent_hash,
            "document_count": len(records),
        }

    def search(self, query: str, top_k: int = 5) -> list[RankedHit]:
        candidate_k = min(len(self.documents), max(top_k * 2, 8))
        return reciprocal_rank_fusion(
            [
                self.sparse.search(query, candidate_k),
                self.dense.search(query, candidate_k),
            ],
            top_k,
        )

    def update(self, new_records: list[dict]):
        current_ids = {row["id"] for row in self.documents}
        new_ids = [row["id"] for row in new_records]
        if current_ids & set(new_ids) or len(new_ids) != len(set(new_ids)):
            raise ValueError("incremental update contains duplicate id")
        return VersionedHybridIndex(
            self.documents + list(new_records),
            generation=self.version["generation"] + 1,
            parent_hash=self.version["corpus_hash"],
        )

In [ ]:
versioned_v1 = VersionedHybridIndex(documents)
new_document = {
    "id": "kb-013",
    "title": "회사 주소 안내",
    "category": "company",
    "content": "본사 주소는 서울시 가상구 학습로 22이며 방문은 사전 예약이 필요합니다.",
    "keywords": ["회사 주소", "본사", "방문"],
}
versioned_v2 = versioned_v1.update([new_document])
assert versioned_v2.version["generation"] == 2
assert versioned_v2.version["parent_hash"] == versioned_v1.version["corpus_hash"]
assert versioned_v2.version["document_count"] == len(documents) + 1
assert versioned_v2.search("회사 본사 주소", 1)[0].document_id == "kb-013"
print("v1=", versioned_v1.version)
print("v2=", versioned_v2.version)

## 8) regression evaluation report

배포 전 후보 index를 고정 평가셋에 돌려 baseline과 비교합니다.
corpus update로 q-013이 이제 answerable해졌으므로 평가 label도
version을 올려 갱신해야 합니다. 여기서는 기존 12개 answerable query에 대한 Recall/MRR 하락을 먼저 gate로 사용합니다.


In [ ]:
def regression_metrics(
    index: VersionedHybridIndex, cases: list[dict], k: int = 3
) -> dict:
    answerable = [case for case in cases if case["relevant_ids"]]
    hits, reciprocal = 0, []
    for case in answerable:
        ids = [hit.document_id for hit in index.search(case["query"], k)]
        ranks = [
            rank
            for rank, doc_id in enumerate(ids, 1)
            if doc_id in set(case["relevant_ids"])
        ]
        hits += bool(ranks)
        reciprocal.append(1 / min(ranks) if ranks else 0.0)
    return {
        "generation": index.version["generation"],
        "corpus_hash": index.version["corpus_hash"],
        "query_count": len(answerable),
        "recall_at_3": hits / len(answerable),
        "mrr": float(np.mean(reciprocal)),
    }


baseline_metrics = regression_metrics(versioned_v1, eval_cases)
candidate_metrics = regression_metrics(versioned_v2, eval_cases)

In [ ]:
def build_regression_report(
    baseline: dict, candidate: dict, tolerance: float = 0.02
) -> dict:
    deltas = {
        metric: candidate[metric] - baseline[metric]
        for metric in ("recall_at_3", "mrr")
    }
    passed = all(delta >= -tolerance for delta in deltas.values())
    return {
        "status": "PASS" if passed else "FAIL",
        "tolerance": tolerance,
        "baseline": baseline,
        "candidate": candidate,
        "deltas": deltas,
    }


regression_report = build_regression_report(baseline_metrics, candidate_metrics)
assert regression_report["status"] in {"PASS", "FAIL"}
print(json.dumps(regression_report, ensure_ascii=False, indent=2))

## 9) query별 회귀 분석

집계 지표가 같아도 서로 다른 query가 좋아지고 나빠질 수 있습니다. baseline/candidate에서 정답 첫 순위를 비교하고,
특히 순위가 하락한 query와 no-answer가 새 문서 추가로 answerable이 된 query를 사람이 검토합니다.


In [ ]:
def first_relevant_rank(index, case: dict, k: int = 5):
    ids = [hit.document_id for hit in index.search(case["query"], k)]
    ranks = [
        rank
        for rank, doc_id in enumerate(ids, 1)
        if doc_id in set(case["relevant_ids"])
    ]
    return min(ranks) if ranks else None


per_query_report = pd.DataFrame(
    {
        "query_id": case["query_id"],
        "query": case["query"],
        "baseline_rank": first_relevant_rank(versioned_v1, case),
        "candidate_rank": first_relevant_rank(versioned_v2, case),
    }
    for case in eval_cases
    if case["relevant_ids"]
)
per_query_report["rank_delta"] = per_query_report["candidate_rank"].fillna(
    99
) - per_query_report["baseline_rank"].fillna(99)
display(per_query_report.sort_values("rank_delta", ascending=False))

## 완료 기준과 확장

- sparse/dense의 score를 직접 섞지 않고 RRF가 필요한 이유를 설명할 수 있다.
- Recall@k/MRR과 no-answer 정확도를 분리하고 threshold의 coverage trade-off를 읽을 수 있다.
- corpus hash/generation/parent hash로 index lineage를 추적할 수 있다.
- 평균 지표뿐 아니라 query별 rank regression을 배포 gate에 포함할 수 있다.

다음 실험: dense SVD dimension, RRF `k`, candidate 폭, threshold를 바꾸되 **평가셋을 보며 규칙을 계속 추가하지 말고**
별도 validation/test split을 만드세요. 실제 embedding/FAISS로 교체해도 이 평가 harness는 그대로 유지할 수 있습니다.
